# Demonstration of news summarizing using voice
This notebook demonstrates the workflow of AI-driven text summarizing using voiced prompting.

The designed workflow is as follows:
<center>

### **Pyaudio** 

record user's voice request into audio data file

&darr;

### **Whisper** 

transcribe the audio file into raw text

&darr;

### **Ollama's search and fetch**

use web_search and web_fetch to retrieve news based on user's request

&darr;

### **Ollama's models** 

summarize the retrieved piece of news into a short read

&darr;

### **Piper TTS**

read the summarized text out loud
</center>


In [1]:
import whisper
import os
import contextlib
with open(os.devnull, "w") as fnull:
    with contextlib.redirect_stderr(fnull):
        import pyaudio
        import speech_recognition as sr
import wave
import sys
from piper import PiperVoice

from ollama import Client
from ollama import chat 

client = Client(headers={'Authorization': f"Bearer {os.getenv('OLLAMA_API_KEY')}"})

key = os.getenv("OLLAMA_API_KEY")

print("Key exists:", key is not None)
print("Key length:", len(key) if key else 0)

Key exists: True
Key length: 57


In [2]:
model = whisper.load_model("base")
result = model.transcribe("../audio_file/test.wav")
print(result['text'])

 Welcome to the world of speech synthesis.


In [3]:

voice = PiperVoice.load("../voice_model/en_GB-jenny_dioco-medium.onnx")
with wave.open("../audio_file/test.wav", "wb") as wav_file:
    voice.synthesize_wav("Welcome to the world of speech synthesis!", wav_file)

sample_rate = voice.config.sample_rate
print(sample_rate)

22050


## Hyperparameter

In [6]:
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 2
RATE = 44100
RECORD_SECONDS = 7
WAVE_OUTPUT_FILENAME = "../audio_file/for_whisper.wav"

##  Code for recording using Pyaudio

Record is saved as "for_whisper.wav"

In [4]:
p = pyaudio.PyAudio()

stream = p.open(channels=CHANNELS, 
                rate=RATE, 
                format=FORMAT, 
                frames_per_buffer=CHUNK, 
                input=True)

print("*** RECORDING ***")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(data)

print("*** Done recording ***")

stream.stop_stream()
stream.close()
p.terminate()

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
wf.setnchannels(CHANNELS)
wf.setsampwidth(p.get_sample_size(FORMAT))
wf.setframerate(RATE)
wf.writeframes(b''.join(frames))
wf.close()

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5705:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM sysdefault
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No

*** RECORDING ***
*** Done recording ***


## Testing Whisper .transcribe()
Model transcribes speech to text.

The audio data is the recording obtained above.

In [7]:
trans = model.transcribe("../audio_file/for_whisper.wav")
print(trans['text'])

 Hello, hello, testing, testing, recording, recording.


## Simple Web search and fetch with Ollama

In [4]:

response = client.web_search("Latest car industry news?")
link = response['results'][0]['url']

summ = client.web_fetch(link)

print(summ['content'])

Toyota to build next-gen EV in China first, with gigacasting tech - Nikkei Asia

# Toyota to build next-gen EV in China first, with gigacasting tech

Automaker diversifies beyond Japan-first approach, targeting global market

Toyota's bZ3X, released in early March, is among its current EV offerings in China. (Photo by Shizuka Tanabe)

Nikkei staff writers

August 28, 2026 06:20 JST

NAGOYA, Japan -- Toyota Motor plans to build a next-generation electric sport utility vehicle in China beginning in fall next year, as global EV competition centers on the world's biggest car market.

## Read Next

#### BYD to employ tech used for Japan-only mini-EV on new European model

#### Nissan, Honda team with self-driving startups to chase US and China rivals

#### China EVs close in on Japan automakers' Australia stronghold, led by BYD

#### Global EV prices drop below hybrids as low-cost Chinese cars spread

#### Toyota aims for 2027 global output of 10.5m vehicles on hybrid demand

#### Toyota's 

## Testing the model's ability to summarize a piece of news
We use Qwen 3 4B in this demo.

The prompt will be **"Summarize this into a short text, keep the key information and do not make up any unmentioned information."**

In [5]:
content = "Summarize this into a short text, keep the key information and do not make up any unmentioned information. \n" + summ['content']
final = chat(model = 'qwen3:4b', messages=[{'role': 'user', 'content': f'{content}'}], stream=True)

text = ""

for i in final:
    text += i['message']['content']
    print(i['message']['content'], end='')

Toyota plans to begin manufacturing its next-generation electric sport utility vehicle in China starting in fall 2027, using gigacasting technology. This move marks a shift from Toyota's traditional Japan-first approach as the automaker targets the global market, with its current bZ3X model already available in China.

## Using text to speech package to read the news out loud

In this demo we use Piper package

In [7]:
p = pyaudio.PyAudio()

stream = p.open(
    format=pyaudio.paInt16,
    channels=1,
    rate=sample_rate,
    output=True,
)

# text = "Hello! This audio is being generated and played directly."

for chunk in voice.synthesize(text):
    stream.write(chunk.audio_int16_bytes)

stream.stop_stream()
stream.close()
p.terminate()

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5705:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM sysdefault
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No